In [ ]:
279
https://github.com/ageron/handson-ml3/blob/main/04_training_linear_models.ipynb

# Вопросы

1. Какой алгоритм обучения линейной регрессии можно использовать, если у вас есть обучающая выборка с миллионами признаков?

2. Предположим, признаки в вашей обучающей выборке имеют очень разные масштабы. Какие алгоритмы могут из-за этого испытывать проблемы и каким образом? Что можно с этим сделать?

3. Может ли градиентный спуск застрять в локальном минимуме при обучении модели логистической регрессии?

4. Все ли алгоритмы градиентного спуска приводят к одной и той же модели, если дать им работать достаточно долго?

5. Предположим, вы используете пакетный градиентный спуск (**Batch Gradient Descent**) и строите график ошибки на валидационной выборке после каждой эпохи. Если вы замечаете, что ошибка на валидационной выборке постоянно увеличивается, что, скорее всего, происходит? Как это исправить?

6. Хорошая ли идея немедленно останавливать мини-пакетный градиентный спуск (**Mini-batch Gradient Descent**), как только ошибка на валидационной выборке увеличивается?

7. Какой алгоритм градиентного спуска (из рассмотренных) быстрее всего достигнет окрестности оптимального решения? Какой из них действительно сойдётся? Как сделать так, чтобы остальные алгоритмы также сошлись?

8. Предположим, вы используете полиномиальную регрессию (**Polynomial Regression**). Вы строите кривые обучения (**learning curves**) и замечаете большую разницу между ошибкой на обучающей выборке и ошибкой на валидационной выборке. Что происходит? Назовите три способа решения этой проблемы.

9. Предположим, вы используете гребневую регрессию (**Ridge Regression**) и замечаете, что ошибка на обучающей и валидационной выборках почти одинаковая и при этом достаточно высокая. Можно ли сказать, что модель страдает от высокого смещения (**high bias**) или высокой дисперсии (**high variance**)? Следует ли увеличить гиперпараметр регуляризации `α` или уменьшить его?

10. Почему вы можете захотеть использовать:

    a. **Ridge Regression** вместо обычной линейной регрессии (то есть без регуляризации)?

    b. **Lasso Regression** вместо Ridge Regression?

    c. **Elastic Net** вместо Lasso Regression?

11. Предположим, вы хотите классифицировать изображения по двум независимым признакам:
    - на улице / в помещении (**outdoor / indoor**);
    - день / ночь (**daytime / nighttime**).

    Следует ли реализовать два классификатора логистической регрессии (**Logistic Regression**) или один классификатор Softmax Regression?

12. Реализуйте пакетный градиентный спуск (**Batch Gradient Descent**) с ранней остановкой (**Early Stopping**) для Softmax Regression без использования Scikit-Learn, только с помощью NumPy. Используйте его для задачи классификации, например, на датасете Iris.

In [ ]:
---

1) Stochastic Gradient Descent (SGD).
Normal Equation становится непрактичным — нужно работать с огромной матрицей и вычислять её обращение.
Batch Gradient Descent тоже может быть дорогим, потому что на каждом шаге проходит по всему training set.
Stochastic Gradient Descent обновляет параметры, используя по одному обучающему примеру, поэтому хорошо подходит для очень больших наборов данных и большого числа признаков

2)
Алгоритмы, использующие Gradient Descent , могут испытывать проблемы, если признаки меют сильно разные
масштабы: Gradient Descent будте сходиться медленнее. Чтобы решить эту проблему, можно масштабировать признаки, например с помощью StandartScaler. Некоторые алгоритмы, такие как Desicion Tree/Random Forest практически не чувствительны к масштабу признаков. 

3)
Нет. Градиентный спуск не может застрять в локальном минимуме при обучении логистической регрессии, потому что функция потерь логистической регрессии является выпуклой. Поэтому у неё нет локальных минимумов, отличающихся от глобального — любой найденный минимум является глобальным. (Тоже самое с линейной)

Linear Regression + MSE → выпуклая функция потерь
мы берём предсказание, вычитаем реальное значение, возводим в квадрат, и число не будет отрицательным никогда.
Гарантия выпуклости — в том, что кривизна этой функции (скорость изменения наклона) — это константа 2, и она сама всегда положительная. Вот именно кривизна, не сама ошибка, доказывает что второй ямы не появится.


Logistic Regression + Log Loss → выпуклая функция потерь
То есть цепочка такая: сигмоида гарантирует диапазон 0-1 → это гарантирует что p×(1-p) никогда не уходит в минус → это гарантирует что кривизна никогда не уходит в минус → это гарантирует что функция выпуклая (одна чаша).


Выпуклость (convex)
       \       /
        \     /
         \   /
          \_/
           ●
Куда бы ты ни положил шарик, он может скатиться в одну и ту же нижнюю точку
---------------

Невыпуклая , как горы (проблема его в том что можно провалиться в локальную глубокую, но возможно есть и другая глубже)
      /\          /\
     /  \        /  \
 ___/    \______/    \___
          ↑
        яма

4)
Нет, не обязательно. Batch GD, SGD и Mini-batch GD обычно сходятся к очень похожему решению для выпуклых задач, таких как линейная регрессия, но делают это с разной скоростью и разной траекторией. SGD из-за случайности может колебаться вокруг минимума. При подходящем уменьшении learning rate он может сойтись к тому же минимуму.

In [ ]:
5) Regularized Linear Models-Early Stopping

Вариант 1 — переобучение
Training error:    ↓ ↓ ↓ ↓ ↓
Validation error:  ↓ ↓ ↗ ↑ ↑
Модель всё лучше подгоняется под train, но всё хуже работает на validation.

→ Overfitting
→ можно использовать early stopping.

Вариант 2 — learning rate слишком большой
Training error:    ↑ ↑ ↑ ↑
Validation error:  ↑ ↑ ↑ ↑
Тут модель вообще не учится нормально. Она перескакивает через минимум.
        ● минимум
      ↙       ↘
   шаг       шаг
      ↘     ↙
Если learning rate слишком большой, каждый шаг может быть настолько большим, что мы постоянно 
перелетаем через минимум.
→ уменьшить learning rate.